In [4]:
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

API_KEY = os.getenv("API_KEY")

if not API_KEY:
    raise ValueError("API_KEY is not set")

print("API Key is set.")

API Key is set.


In [5]:
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash",temperature=0,api_key=os.environ['API_KEY'])

In [6]:


## Your text data
JD = """# Software & Systems Engineer

**Organization:** Defence Research & Technology Division  
**Role:** Software & Systems Engineer  
**Experience:** 0–3 years

## Job Description

Work on software systems supporting simulation, telemetry processing, automation, and mission-oriented engineering applications.

### Responsibilities
- Develop software using Python/C++.
- Design and implement reliable software components.
- Work with Linux-based systems and Git.
- Develop APIs and data-processing utilities.
- Debug performance and reliability issues.
- Write unit and integration tests.
- Work with engineers to translate system requirements into software solutions.
- Participate in technical design, testing, and documentation.

### Required Skills
- Strong programming fundamentals.
- Python or C++.
- Data structures and algorithms.
- Object-oriented programming.
- SQL and database fundamentals.
- Linux fundamentals.
- Git/version control.
- Debugging and problem-solving.

### Preferred Skills
- Multithreading and concurrency.
- Distributed systems.
- Docker.
- REST APIs.
- Simulation systems.
- Telemetry/data processing.

### Evaluation Areas
Programming fundamentals, problem-solving, software design, debugging, systems knowledge, and practical engineering judgment.
""" 


In [8]:
resume_path = "./Sample_Files/DRDO_Sample_Resume_1.pdf"

loader = PyPDFLoader(resume_path)
resume_docs = loader.load()

resume_text = "\n".join(
    doc.page_content for doc in resume_docs
)

In [15]:
question_df = pd.read_csv(
    "./Sample_Files/DRDO_150_Question_Bank.csv")


In [16]:
question_docs = []

for _, row in question_df.iterrows():

    metadata = {
        "question_id": row["question_id"],
        "domain": row["domain"],
        "topic": row["topic"],
        "difficulty": row["difficulty"],
        "question_type": row["question_type"],
        "source": row["source"],
        "language": row["language"],
        "content_type": row["content_type"]
    }

    question_docs.append(
        Document(
            page_content=row["question"],
            metadata=metadata
        )
    )

In [ ]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    api_key=API_KEY
)


3072
[-0.030082783, 0.005711784, 0.019011842, 0.01106112, 0.0044799233]


In [21]:
vectorstore = Chroma.from_documents(
    documents=question_docs,
    embedding=embedding_model,
    collection_name="interview_question_bank"
)


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-2\nPlease retry in 35.132324465s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'q

In [20]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

retrieval_query = f"""
Candidate Resume:
{resume_text}

Job Description:
{JD}

Find interview questions relevant to this candidate and this role.
Focus on the candidate's skills, experience, and overlap with the job requirements.
"""

retrieved_docs = retriever.invoke(retrieval_query)

print("Retrieved:", len(retrieved_docs))

for doc in retrieved_docs:
    print(
        doc.metadata["question_id"],
        "|",
        doc.metadata["domain"],
        "|",
        doc.metadata["topic"],
        "|",
        doc.metadata["difficulty"],
        "|",
        doc.page_content
    )

NameError: name 'vectorstore' is not defined

In [ ]:
retrieval_query = f"""
Candidate Resume:
{resume_text}

Job Description:
{JD}

Find interview questions relevant to this candidate and this role.
Focus on the candidate's skills, experience, and overlap with the job requirements.
"""

In [ ]:
retrieved_docs = retriever.invoke(retrieval_query)

for doc in retrieved_docs:
    print(
        doc.metadata["question_id"],
        "|",
        doc.metadata["topic"],
        "|",
        doc.page_content
    )

In [ ]:
retrieved_questions = "\n".join(
    f"- [{doc.metadata['question_id']}] "
    f"{doc.page_content}"
    for doc in retrieved_docs
)

prompt = f"""
You are a technical interview question selector.

Analyze the candidate's resume against the provided job description.

Candidate Resume:
{resume_text}

Job Description:
{JD}

Retrieved Question Bank:
{retrieved_questions}

Generate the 10 most relevant interview questions for this candidate.

Rules:
1. Questions must be relevant to the JD.
2. Questions should match the candidate's actual skills and experience.
3. Prefer areas where the candidate and JD overlap.
4. Progress from fundamentals to deeper technical questions.
5. Include practical/scenario-based questions where appropriate.
6. Do not invent technologies that are absent from both the candidate profile and JD.
7. Return only the 10 questions, numbered 1–10.
"""

In [ ]:
response = llm.invoke(prompt)

print(response.content)